In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from pathlib import Path
from time import perf_counter
import gc
import shutil
import subprocess
import threading
import time

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers.models.siglip.configuration_siglip import SiglipVisionConfig
from transformers.models.siglip.modeling_siglip import SiglipMultiheadAttentionPoolingHead


In [ ]:
# Constants.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "benchmark" else Path.cwd()
DATA_CSV = PROJECT_ROOT / "artifacts/processed_data/chexpertplus_frontal_5labels.csv"
LINEAR_METRICS_CSV = PROJECT_ROOT / "artifacts/probing/linear_probe/results/experiment_metrics.csv"
TEMP_CACHE_DIR = PROJECT_ROOT / "temp/mha_activation_cache"

MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"
MODEL_DTYPE = torch.bfloat16
RANDOM_STATE = 42

TARGET_LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
BENCHMARK_LABEL = "Pleural Effusion"
BENCHMARK_PROMPT_ORDER = "text_first"
LABEL_SLUG = BENCHMARK_LABEL.lower().replace(" ", "_")
ACTIVATION_DIR = TEMP_CACHE_DIR / BENCHMARK_PROMPT_ORDER / LABEL_SLUG
TEST_LAYER_BLOCKS = [list(range(0, 5)), list(range(5, 10))]
TEST_LAYERS = [layer for block in TEST_LAYER_BLOCKS for layer in block]

MEDGEMMA_IMAGE_SIZE = 896
MEDGEMMA_BATCH_SIZE = 24
MHA_BATCH_SIZE = 512
MHA_EPOCHS = 30
MHA_EVAL_EVERY_EPOCHS = 10
MHA_LEARNING_RATE = 5e-5
MHA_WEIGHT_DECAY = 1e-4
MHA_NUM_HEADS = 20
MHA_MLP_DIM = 10240
HDF5_IO_CHUNK_ROWS = 16

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Small utilities for images and resource logging.

def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def layer_file(layer):
    return ACTIVATION_DIR / f"layer_{layer:02d}.h5"


def process_rss_gb():
    status_path = Path("/proc/self/status")
    if not status_path.exists():
        return np.nan
    for line in status_path.read_text().splitlines():
        if line.startswith("VmRSS:"):
            return float(line.split()[1]) / 1024**2
    return np.nan


def system_ram_gb():
    meminfo_path = Path("/proc/meminfo")
    if not meminfo_path.exists():
        return {"system_ram_used_gb": np.nan, "system_ram_free_gb": np.nan}
    vals = {}
    for line in meminfo_path.read_text().splitlines():
        key, value = line.split(":", 1)
        vals[key] = float(value.split()[0]) / 1024**2
    return {
        "system_ram_used_gb": vals.get("MemTotal", np.nan) - vals.get("MemAvailable", np.nan),
        "system_ram_free_gb": vals.get("MemAvailable", np.nan),
    }


def disk_free_gb(path):
    path = Path(path)
    if not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free / 1024**3


def query_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        util, mem_used, mem_total = [float(x.strip()) for x in out.split(",")]
        return {"gpu_util_pct": util, "gpu_mem_used_mb": mem_used, "gpu_mem_total_mb": mem_total}
    except Exception:
        return {"gpu_util_pct": np.nan, "gpu_mem_used_mb": np.nan, "gpu_mem_total_mb": np.nan}


class GpuMonitor:
    def __init__(self, interval_sec=5):
        self.interval_sec = interval_sec
        self.rows = []
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self.stop.is_set():
            self.rows.append(query_gpu())
            time.sleep(self.interval_sec)

    def __enter__(self):
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.stop.set()
        self.thread.join()

    def summary(self, prefix):
        df_gpu = pd.DataFrame(self.rows)
        if df_gpu.empty:
            return {f"{prefix}_avg_gpu_util_pct": np.nan, f"{prefix}_max_gpu_util_pct": np.nan, f"{prefix}_max_gpu_mem_mb": np.nan}
        return {
            f"{prefix}_avg_gpu_util_pct": df_gpu["gpu_util_pct"].mean(),
            f"{prefix}_max_gpu_util_pct": df_gpu["gpu_util_pct"].max(),
            f"{prefix}_max_gpu_mem_mb": df_gpu["gpu_mem_used_mb"].max(),
        }


def resource_row(stage):
    row = {"stage": stage, "process_rss_gb": process_rss_gb(), "disk_free_gb": disk_free_gb(TEMP_CACHE_DIR)}
    row.update(system_ram_gb())
    row.update(query_gpu())
    return row


def show_resource(stage):
    display(pd.DataFrame([resource_row(stage)]))


In [ ]:
# Load data and existing linear-probe metrics.

df = pd.read_csv(DATA_CSV)
labels = (df[TARGET_LABELS].to_numpy() == 1).astype(np.int8)
y_np = labels[:, TARGET_LABELS.index(BENCHMARK_LABEL)].astype(np.float32)
train_idx = np.where(df["probe_split"].to_numpy() == "train")[0]
test_idx = np.where(df["probe_split"].to_numpy() == "test")[0]
image_paths = df["image_path"].tolist()
label_i = TARGET_LABELS.index(BENCHMARK_LABEL)

linear_metrics = pd.read_csv(LINEAR_METRICS_CSV)
if "model_name" not in linear_metrics.columns:
    linear_metrics["model_name"] = np.where(linear_metrics["feature"].eq("medsiglip_standalone"), "medsiglip_standalone", "base_medgemma")
linear_metrics["layer"] = pd.to_numeric(linear_metrics["layer"], errors="coerce")

comparison_rows = []
medsiglip_df = linear_metrics[(linear_metrics["feature"] == "medsiglip_standalone") & (linear_metrics["label"] == BENCHMARK_LABEL)]
if not medsiglip_df.empty:
    row = medsiglip_df.iloc[0]
    comparison_rows.append({"source": "old_metrics", "probe": "MedSigLIP baseline", "layer": np.nan, "auroc": row["auroc"], "auprc": row["auprc"], "positive_prevalence": row["positive_prevalence"]})

old_linear_df = linear_metrics[
    (linear_metrics["model_name"] == "base_medgemma")
    & (linear_metrics["prompt_order"] == BENCHMARK_PROMPT_ORDER)
    & (linear_metrics["label"] == BENCHMARK_LABEL)
    & (linear_metrics["feature"].isin(["medgemma_layer_mean_image_token", "medgemma_layer_last_image_token"]))
    & (linear_metrics["layer"].isin(TEST_LAYERS))
]
for _, row in old_linear_df.iterrows():
    comparison_rows.append({
        "source": "old_metrics",
        "probe": row["feature"],
        "layer": int(row["layer"]),
        "auroc": row["auroc"],
        "auprc": row["auprc"],
        "positive_prevalence": row["positive_prevalence"],
    })

print("rows", len(df), "train", len(train_idx), "test", len(test_idx))
print("benchmark", BENCHMARK_PROMPT_ORDER, BENCHMARK_LABEL, "test layers", TEST_LAYERS)
display(pd.DataFrame(comparison_rows).sort_values(["layer", "probe"], na_position="first"))
show_resource("after_load_data")


In [ ]:
# Load MedGemma processor and cache processed pixel_values without caching all PIL images.

medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
medgemma_pixel_values = torch.empty(
    (len(df), 3, MEDGEMMA_IMAGE_SIZE, MEDGEMMA_IMAGE_SIZE),
    dtype=MODEL_DTYPE,
    device="cpu",
)

show_resource("before_image_processing")
image_process_start = perf_counter()
for start in tqdm(range(0, len(df), MEDGEMMA_BATCH_SIZE), desc="Process images"):
    end = min(start + MEDGEMMA_BATCH_SIZE, len(df))
    images = [load_rgb(path) for path in image_paths[start:end]]
    pixel_inputs = medgemma_processor.image_processor(images=images, return_tensors="pt", do_pan_and_scan=False)
    medgemma_pixel_values[start:end].copy_(pixel_inputs["pixel_values"].to(dtype=MODEL_DTYPE))
    del images, pixel_inputs
    gc.collect()
image_process_sec = perf_counter() - image_process_start

print("cached pixel_values", tuple(medgemma_pixel_values.shape), medgemma_pixel_values.dtype)
print("image_process_sec", round(image_process_sec, 2))
show_resource("after_image_processing")


In [ ]:
# Prompt/token helpers.

def prompt_text(prompt_order, label):
    question = f"Question: Is there {label.lower()} in this image? Answer yes or no."
    image_item = {"type": "image"}
    if prompt_order == "image_first":
        content = [image_item, {"type": "text", "text": f"\n{question}\nAnswer: "}]
    else:
        content = [{"type": "text", "text": f"{question}\n"}, image_item, {"type": "text", "text": "\nAnswer: "}]
    messages = [{"role": "user", "content": content}]
    text = medgemma_processor.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
    if isinstance(text, list):
        text = text[0]
    return text.replace(medgemma_processor.boi_token, medgemma_processor.full_image_sequence)


def tokenize_prompt(prompt_order, label, batch_size):
    inputs = medgemma_processor.tokenizer(
        [prompt_text(prompt_order, label)] * batch_size,
        return_tensors="pt",
        padding=True,
    )
    if hasattr(medgemma_processor, "create_mm_token_type_ids"):
        token_type_ids = medgemma_processor.create_mm_token_type_ids(inputs["input_ids"])
        if not torch.is_tensor(token_type_ids):
            token_type_ids = torch.tensor(token_type_ids)
        inputs["token_type_ids"] = token_type_ids
    return inputs


In [ ]:
# Run base MedGemma once and save all 34 image-token activation layers to HDF5.

device = "cuda" if torch.cuda.is_available() else "cpu"
medgemma_model = AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL_ID, dtype=MODEL_DTYPE, device_map="auto").eval()
model_device = next(medgemma_model.parameters()).device
model_config = medgemma_model.config
num_layers = model_config.text_config.num_hidden_layers
hidden_size = model_config.text_config.hidden_size
image_token_id = getattr(model_config, "image_token_id", None)
if image_token_id is None:
    image_token_id = getattr(model_config, "image_token_index")

check_inputs = tokenize_prompt(BENCHMARK_PROMPT_ORDER, BENCHMARK_LABEL, 1)
image_token_count = int(check_inputs["input_ids"].eq(image_token_id).sum().item())
all_layers = list(range(num_layers))
expected_layer_gb = len(df) * image_token_count * hidden_size * 4 / 1024**3
expected_total_gb = expected_layer_gb * num_layers

for old_file in ACTIVATION_DIR.glob("layer_*.h5"):
    old_file.unlink()

print("num_layers", num_layers, "hidden_size", hidden_size, "image_token_count", image_token_count)
print("activation_dir", ACTIVATION_DIR)
print("expected_layer_file_gb", round(expected_layer_gb, 2))
print("expected_total_file_gb", round(expected_total_gb, 2))
show_resource("before_activation_extraction")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

h5_files = []
dsets = {}
extract_start = perf_counter()
try:
    for layer in all_layers:
        h5 = h5py.File(layer_file(layer), "w")
        h5_files.append(h5)
        dsets[layer] = h5.create_dataset(
            "image_tokens",
            shape=(len(df), image_token_count, hidden_size),
            dtype="float32",
            chunks=(HDF5_IO_CHUNK_ROWS, image_token_count, hidden_size),
        )

    with GpuMonitor() as gpu_monitor:
        for start in tqdm(range(0, len(df), MEDGEMMA_BATCH_SIZE), desc="Extract all layers"):
            end = min(start + MEDGEMMA_BATCH_SIZE, len(df))
            batch_size = end - start
            inputs = tokenize_prompt(BENCHMARK_PROMPT_ORDER, BENCHMARK_LABEL, batch_size)
            inputs["pixel_values"] = medgemma_pixel_values[start:end]
            inputs = {
                k: v.to(device=model_device, dtype=MODEL_DTYPE) if v.is_floating_point() else v.to(device=model_device)
                for k, v in inputs.items()
            }

            with torch.inference_mode():
                outputs = medgemma_model(
                    **inputs,
                    output_hidden_states=True,
                    use_cache=False,
                    logits_to_keep=1,
                    return_dict=True,
                )

            image_mask = inputs["input_ids"].eq(image_token_id)
            for layer in all_layers:
                hidden = outputs.hidden_states[layer + 1]
                tokens = hidden[image_mask].reshape(batch_size, image_token_count, hidden_size)
                dsets[layer][start:end] = tokens.detach().to("cpu", dtype=torch.float32).numpy()
                del hidden, tokens

            del outputs, inputs
finally:
    for h5 in h5_files:
        h5.flush()
        h5.close()

activation_extract_sec = perf_counter() - extract_start
extract_gpu = gpu_monitor.summary("extract")
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else np.nan

check_rows = []
for layer in all_layers:
    with h5py.File(layer_file(layer), "r") as h5:
        dset = h5["image_tokens"]
        check_rows.append({
            "layer": layer,
            "shape": tuple(dset.shape),
            "dtype": str(dset.dtype),
            "file_size_gb": layer_file(layer).stat().st_size / 1024**3,
        })

print("activation_extract_sec", round(activation_extract_sec, 2))
print("torch_peak_gpu_gb", round(peak_gpu_gb, 2) if not np.isnan(peak_gpu_gb) else peak_gpu_gb)
print("saved layer files", len(check_rows))
display(pd.DataFrame(check_rows).head())
display(pd.DataFrame([{**extract_gpu, "activation_extract_sec": activation_extract_sec, "torch_peak_gpu_gb": peak_gpu_gb, **resource_row("after_activation_extraction")}]))

del medgemma_model, medgemma_pixel_values
gc.collect()
torch.cuda.empty_cache()
show_resource("after_deleting_medgemma_and_pixels")


In [ ]:
# Multi-head attention pooling probe and training helpers.

class MHAPoolingProbe(torch.nn.Module):
    def __init__(self, hidden_size=2560, intermediate_size=10240, num_attention_heads=20):
        super().__init__()
        cfg = SiglipVisionConfig(
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            num_attention_heads=num_attention_heads,
            layer_norm_eps=1e-6,
            hidden_act="gelu_pytorch_tanh",
        )
        self.pooler = SiglipMultiheadAttentionPoolingHead(cfg)
        self.classifier = torch.nn.Linear(hidden_size, 1)

    def forward(self, x):
        pooled = self.pooler(x)
        return self.classifier(pooled).squeeze(-1)


def predict_mha_scores(probe, tokens, indices, device):
    scores = []
    probe.eval()
    autocast_device = "cuda" if device == "cuda" else "cpu"
    with torch.inference_mode():
        for start in range(0, len(indices), MHA_BATCH_SIZE):
            idx = indices[start:start + MHA_BATCH_SIZE]
            x_batch = tokens[idx].to(device=device, non_blocking=True)
            with torch.autocast(device_type=autocast_device, dtype=torch.bfloat16, enabled=device == "cuda"):
                logits = probe(x_batch)
            scores.append(logits.float().cpu())
            del x_batch, logits
    return torch.cat(scores).numpy()


def train_mha_probe_sequential(tokens, layer):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    y = torch.tensor(y_np, dtype=torch.float32)
    train_indices = torch.tensor(train_idx, dtype=torch.long)
    test_indices = torch.tensor(test_idx, dtype=torch.long)
    y_train = y_np[train_idx]
    y_test = y_np[test_idx]

    probe = MHAPoolingProbe(tokens.shape[-1], MHA_MLP_DIM, MHA_NUM_HEADS).to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=MHA_LEARNING_RATE, weight_decay=MHA_WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    autocast_device = "cuda" if device == "cuda" else "cpu"

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    train_start = perf_counter()
    history = []
    with GpuMonitor() as gpu_monitor:
        for epoch in range(MHA_EPOCHS):
            perm = train_indices[torch.randperm(len(train_indices))]
            epoch_losses = []
            probe.train()
            for start in range(0, len(perm), MHA_BATCH_SIZE):
                idx = perm[start:start + MHA_BATCH_SIZE]
                x_batch = tokens[idx].to(device=device, non_blocking=True)
                y_batch = y[idx].to(device=device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=autocast_device, dtype=torch.bfloat16, enabled=device == "cuda"):
                    logits = probe(x_batch)
                loss = criterion(logits.float(), y_batch)
                loss.backward()
                optimizer.step()

                epoch_losses.append(float(loss.detach().cpu()))
                del x_batch, y_batch, logits, loss

            train_loss = float(np.mean(epoch_losses))
            train_auroc = np.nan
            test_auroc = np.nan
            if (epoch + 1) == 1 or (epoch + 1) % MHA_EVAL_EVERY_EPOCHS == 0 or (epoch + 1) == MHA_EPOCHS:
                train_scores = predict_mha_scores(probe, tokens, train_indices, device)
                test_scores = predict_mha_scores(probe, tokens, test_indices, device)
                train_auroc = float(roc_auc_score(y_train, train_scores))
                test_auroc = float(roc_auc_score(y_test, test_scores))
                print("sequential layer", layer, "epoch", epoch + 1, "loss", round(train_loss, 4), "test_auroc", round(test_auroc, 4))
            history.append({"training_mode": "sequential", "layer": layer, "epoch": epoch + 1, "train_loss": train_loss, "train_auroc": train_auroc, "test_auroc": test_auroc})

        final_scores = predict_mha_scores(probe, tokens, test_indices, device)

    train_sec = perf_counter() - train_start
    best_eval = pd.DataFrame(history).dropna(subset=["test_auroc"]).sort_values("test_auroc", ascending=False).iloc[0]
    peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else np.nan

    del probe, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "source": "fresh_mha_benchmark",
        "training_mode": "sequential",
        "probe": "mha_pooled_image_token",
        "layer": layer,
        "auroc": float(roc_auc_score(y_test, final_scores)),
        "auprc": float(average_precision_score(y_test, final_scores)),
        "positive_prevalence": float(y_test.mean()),
        "mha_train_sec": train_sec,
        "final_train_loss": history[-1]["train_loss"],
        "best_epoch": int(best_eval["epoch"]),
        "best_auroc": float(best_eval["test_auroc"]),
        "torch_peak_gpu_gb": peak_gpu_gb,
        "eval_history": history,
        **gpu_monitor.summary(f"sequential_layer_{layer}"),
        **resource_row(f"after_sequential_layer_{layer}"),
    }


In [ ]:
# Joint-loss training keeps one independent MHA probe per layer.

def train_mha_probe_joint(layer_caches, layer_block):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    y = torch.tensor(y_np, dtype=torch.float32)
    train_indices = torch.tensor(train_idx, dtype=torch.long)
    test_indices = torch.tensor(test_idx, dtype=torch.long)
    y_train = y_np[train_idx]
    y_test = y_np[test_idx]
    hidden_size = next(iter(layer_caches.values())).shape[-1]

    probes = torch.nn.ModuleDict({str(layer): MHAPoolingProbe(hidden_size, MHA_MLP_DIM, MHA_NUM_HEADS) for layer in layer_block}).to(device)
    optimizer = torch.optim.AdamW(probes.parameters(), lr=MHA_LEARNING_RATE, weight_decay=MHA_WEIGHT_DECAY)
    criterion = torch.nn.BCEWithLogitsLoss()
    autocast_device = "cuda" if device == "cuda" else "cpu"

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    train_start = perf_counter()
    history = []
    with GpuMonitor() as gpu_monitor:
        for epoch in range(MHA_EPOCHS):
            perm = train_indices[torch.randperm(len(train_indices))]
            epoch_losses = {layer: [] for layer in layer_block}
            probes.train()
            for start in range(0, len(perm), MHA_BATCH_SIZE):
                idx = perm[start:start + MHA_BATCH_SIZE]
                y_batch = y[idx].to(device=device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                total_loss = 0.0

                for layer in layer_block:
                    x_batch = layer_caches[layer][idx].to(device=device, non_blocking=True)
                    with torch.autocast(device_type=autocast_device, dtype=torch.bfloat16, enabled=device == "cuda"):
                        logits = probes[str(layer)](x_batch)
                    loss = criterion(logits.float(), y_batch)
                    total_loss = total_loss + loss
                    epoch_losses[layer].append(float(loss.detach().cpu()))
                    del x_batch, logits

                total_loss.backward()
                optimizer.step()
                del y_batch, total_loss

            eval_scores = {}
            if (epoch + 1) == 1 or (epoch + 1) % MHA_EVAL_EVERY_EPOCHS == 0 or (epoch + 1) == MHA_EPOCHS:
                for layer in layer_block:
                    train_scores = predict_mha_scores(probes[str(layer)], layer_caches[layer], train_indices, device)
                    test_scores = predict_mha_scores(probes[str(layer)], layer_caches[layer], test_indices, device)
                    eval_scores[layer] = (float(roc_auc_score(y_train, train_scores)), float(roc_auc_score(y_test, test_scores)))
                print("joint epoch", epoch + 1, "mean_loss", round(np.mean([np.mean(v) for v in epoch_losses.values()]), 4), "test_auroc", {layer: round(eval_scores[layer][1], 4) for layer in layer_block})

            for layer in layer_block:
                train_auroc, test_auroc = eval_scores.get(layer, (np.nan, np.nan))
                history.append({"training_mode": "joint_loss", "layer": layer, "epoch": epoch + 1, "train_loss": float(np.mean(epoch_losses[layer])), "train_auroc": train_auroc, "test_auroc": test_auroc})

        final_scores = {layer: predict_mha_scores(probes[str(layer)], layer_caches[layer], test_indices, device) for layer in layer_block}

    train_sec = perf_counter() - train_start
    history_df = pd.DataFrame(history)
    peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else np.nan
    rows = []
    for layer in layer_block:
        layer_history = history_df[history_df["layer"] == layer]
        best_eval = layer_history.dropna(subset=["test_auroc"]).sort_values("test_auroc", ascending=False).iloc[0]
        rows.append({
            "source": "fresh_mha_benchmark",
            "training_mode": "joint_loss",
            "probe": "mha_pooled_image_token",
            "layer": layer,
            "auroc": float(roc_auc_score(y_test, final_scores[layer])),
            "auprc": float(average_precision_score(y_test, final_scores[layer])),
            "positive_prevalence": float(y_test.mean()),
            "mha_train_sec": train_sec,
            "final_train_loss": float(layer_history.iloc[-1]["train_loss"]),
            "best_epoch": int(best_eval["epoch"]),
            "best_auroc": float(best_eval["test_auroc"]),
            "torch_peak_gpu_gb": peak_gpu_gb,
            **gpu_monitor.summary("joint_loss"),
            **resource_row(f"after_joint_loss_layer_{layer}"),
        })

    del probes, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return rows, history


In [ ]:
# Load one block of layer activations from disk.

def load_layer_block(layer_block):
    layer_caches = {}
    read_rows = []
    for layer in layer_block:
        read_start = perf_counter()
        with h5py.File(layer_file(layer), "r") as h5:
            dset = h5["image_tokens"]
            tokens = torch.empty(dset.shape, dtype=MODEL_DTYPE, device="cpu")
            for start in tqdm(range(0, len(df), HDF5_IO_CHUNK_ROWS), desc=f"Read layer {layer}"):
                end = min(start + HDF5_IO_CHUNK_ROWS, len(df))
                tokens[start:end].copy_(torch.from_numpy(dset[start:end]).to(dtype=MODEL_DTYPE))
        read_sec = perf_counter() - read_start
        layer_caches[layer] = tokens
        read_rows.append({"layer": layer, "read_sec": read_sec, "ram_cache_gb": tokens.numel() * 2 / 1024**3})
    display(pd.DataFrame(read_rows))
    return layer_caches


In [ ]:
# Train sequential and joint-loss MHA probes for two 5-layer blocks.

mha_rows = []
history_rows = []
block_resource_rows = []

for layer_block in TEST_LAYER_BLOCKS:
    print("starting layer block", layer_block, "display layers", [layer + 1 for layer in layer_block])
    block_resource_rows.append(resource_row(f"before_loading_block_{layer_block[0]}_{layer_block[-1]}"))
    show_resource(f"before_loading_block_{layer_block[0]}_{layer_block[-1]}")

    layer_caches = load_layer_block(layer_block)
    block_resource_rows.append(resource_row(f"after_loading_block_{layer_block[0]}_{layer_block[-1]}"))
    show_resource(f"after_loading_block_{layer_block[0]}_{layer_block[-1]}")

    for layer in layer_block:
        print("sequential training layer", layer)
        row = train_mha_probe_sequential(layer_caches[layer], layer)
        history_rows.extend(row.pop("eval_history"))
        mha_rows.append(row)

    block_resource_rows.append(resource_row(f"after_sequential_block_{layer_block[0]}_{layer_block[-1]}"))
    show_resource(f"after_sequential_block_{layer_block[0]}_{layer_block[-1]}")

    print("joint-loss training block", layer_block)
    joint_rows, joint_history = train_mha_probe_joint(layer_caches, layer_block)
    mha_rows.extend(joint_rows)
    history_rows.extend(joint_history)

    block_resource_rows.append(resource_row(f"after_joint_block_{layer_block[0]}_{layer_block[-1]}"))
    show_resource(f"after_joint_block_{layer_block[0]}_{layer_block[-1]}")

    del layer_caches
    gc.collect()
    torch.cuda.empty_cache()
    block_resource_rows.append(resource_row(f"after_deleting_block_{layer_block[0]}_{layer_block[-1]}"))
    show_resource(f"after_deleting_block_{layer_block[0]}_{layer_block[-1]}")

mha_df = pd.DataFrame(mha_rows)
history_df = pd.DataFrame(history_rows)
comparison_df = pd.concat([pd.DataFrame(comparison_rows), mha_df], ignore_index=True, sort=False)

result_cols = [
    "source",
    "training_mode",
    "probe",
    "layer",
    "auroc",
    "auprc",
    "positive_prevalence",
    "best_auroc",
    "best_epoch",
    "mha_train_sec",
    "torch_peak_gpu_gb",
]
resource_cols = [
    "stage",
    "process_rss_gb",
    "disk_free_gb",
    "system_ram_used_gb",
    "system_ram_free_gb",
    "gpu_util_pct",
    "gpu_mem_used_mb",
    "gpu_mem_total_mb",
]

print("MHA rows", len(mha_df))
display(comparison_df[[col for col in result_cols if col in comparison_df.columns]].sort_values(["layer", "probe", "training_mode"], na_position="first"))
display(pd.DataFrame(block_resource_rows)[resource_cols])

In [ ]:
# Delete temporary activation files after the benchmark.

print("deleting activation files", ACTIVATION_DIR)
if ACTIVATION_DIR.exists():
    shutil.rmtree(ACTIVATION_DIR)
gc.collect()
torch.cuda.empty_cache()
show_resource("after_deleting_activation_files")